# SQL Generator with Verifier | Generator-Verifier (Generator-Discriminator)

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke
import sqlite3
import tempfile
import os

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

class GenVerifyState(TypedDict):
    question: str
    schema: str
    sql: NotRequired[str]
    verification_result: NotRequired[str]
    iteration: NotRequired[int]
    final_answer: NotRequired[str]

MAX_ITERATIONS = 3

# Create a temporary SQLite database with sample data
DB_PATH = os.path.join(tempfile.gettempdir(), "sample_genverify.db")

def setup_database():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.executescript("""
        CREATE TABLE IF NOT EXISTS employees (
            id INTEGER PRIMARY KEY, name TEXT, department TEXT, salary REAL, hire_date TEXT
        );
        DELETE FROM employees;
        INSERT INTO employees VALUES (1, 'Alice', 'Engineering', 120000, '2021-03-15');
        INSERT INTO employees VALUES (2, 'Bob', 'Engineering', 115000, '2022-01-10');
        INSERT INTO employees VALUES (3, 'Carol', 'Marketing', 95000, '2020-06-01');
        INSERT INTO employees VALUES (4, 'Dave', 'Marketing', 90000, '2023-02-20');
        INSERT INTO employees VALUES (5, 'Eve', 'Sales', 85000, '2021-11-05');
        INSERT INTO employees VALUES (6, 'Frank', 'Sales', 88000, '2022-08-15');
        INSERT INTO employees VALUES (7, 'Grace', 'Engineering', 130000, '2019-04-22');
    """)
    conn.commit()
    conn.close()

setup_database()

SCHEMA = """
TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary REAL,
    hire_date TEXT  -- format: YYYY-MM-DD
)
"""

In [4]:
def generate_sql(state: GenVerifyState) -> dict:
    context = ""
    if state.get("verification_result") and "Error" in state.get("verification_result", ""):
        context = f"\n\nPrevious attempt failed:\nSQL: {state['sql']}\nError: {state['verification_result']}\n\nFix the SQL."
    response = model.invoke(
        f"Generate a SQLite SQL query to answer this question.\n\n"
        f"Schema:\n{state['schema']}\n\n"
        f"Question: {state['question']}{context}\n\n"
        f"Return ONLY the SQL query, no explanation or markdown."
    )
    sql = response.content.strip()
    if sql.startswith("```"):
        sql = sql.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    return {"sql": sql, "iteration": state.get("iteration", 0) + 1}

In [5]:
def verify_sql(state: GenVerifyState) -> Command[Literal["generate_sql", "finalize"]]:
    """Verify by actually executing the SQL, then route via Command."""
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(state["sql"])
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        conn.close()
        if not rows:
            verification_result = f"Query returned no results. Columns: {columns}"
        else:
            header = " | ".join(columns)
            row_strs = [" | ".join(str(v) for v in row) for row in rows]
            result = f"{header}\n" + "\n".join(row_strs)
            verification_result = f"Success:\n{result}"
    except Exception as e:
        verification_result = f"Error: {str(e)}"
    iteration = state.get("iteration", 0)
    if verification_result.startswith("Success") or iteration >= MAX_ITERATIONS:
        return Command(goto="finalize", update={"verification_result": verification_result})
    return Command(goto="generate_sql", update={"verification_result": verification_result})

In [6]:
def finalize(state: GenVerifyState) -> dict:
    return {"final_answer": f"SQL:\n{state['sql']}\n\nResult:\n{state['verification_result']}"}

In [7]:
graph = StateGraph(GenVerifyState)
graph.add_node("generate_sql", generate_sql)
graph.add_node("verify_sql", verify_sql)
graph.add_node("finalize", finalize)

graph.add_edge(START, "generate_sql")
graph.add_edge("generate_sql", "verify_sql")
# No add_conditional_edges needed -- verify_sql returns Command to route directly
graph.add_edge("finalize", END)

gen_verify = graph.compile()

In [8]:
plot_mermaid(gen_verify)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate_sql(generate_sql)
	verify_sql(verify_sql)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate_sql;
	generate_sql --> verify_sql;
	verify_sql -.-> finalize;
	verify_sql -.-> generate_sql;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [9]:
result = gen_verify.invoke({
    "question": "What is the average salary per department, ordered by average salary descending?",
    "schema": SCHEMA,
})
print(result["final_answer"])

SQL:
SELECT department, AVG(salary) AS average_salary
FROM employees
GROUP BY department
ORDER BY average_salary DESC;

Result:
Success:
department | average_salary
Engineering | 121666.66666666667
Marketing | 92500.0
Sales | 86500.0


In [10]:
# Streaming

stream_invoke(
    gen_verify, {
        "question": "What is the average salary per department, ordered by average salary descending?",
        "schema": SCHEMA,
    }
)


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'question': 'What is the average salary per department, ordered by average salary descending?',
 'schema': '\nTABLE employees (\n    id INTEGER PRIMARY KEY,\n    name TEXT,\n    department TEXT,\n    salary REAL,\n    hire_date TEXT  -- format: YYYY-MM-DD\n)\n',
 'sql': 'SELECT department, AVG(salary) AS average_salary\nFROM employees\nGROUP BY department\nORDER BY average_salary DESC;',
 'verification_result': 'Success:\ndepartment | average_salary\nEngineering | 121666.66666666667\nMarketing | 92500.0\nSales | 86500.0',
 'iteration': 1,
 'final_answer': 'SQL:\nSELECT department, AVG(salary) AS average_salary\nFROM employees\nGROUP BY department\nORDER BY average_salary DESC;\n\nResult:\nSuccess:\ndepartment | average_salary\nEngineering | 121666.66666666667\nMarketing | 92500.0\nSales | 86500.0'}